# DMS benchmark: MediaPipe + Claude vs fixed EAR threshold

Offline evaluation on pre-recorded video: extract **EAR / MAR / head pose** per frame (Python MediaPipe Face Mesh), run **Claude** alert reasoning on rolling summaries (same idea as production `AlertEngine`), and compare to a **fixed-threshold baseline** (mean EAR < 0.21 sustained for ≥1s).

**Outputs:** JSON event log, summary stats, matplotlib overlays.

**Video sources (see README):** NTHU-DDD (agreement), UTA-RLDD / Kaggle, or your own MP4 via upload or Google Drive.

## 1. Install dependencies

In [ ]:
%pip install -q mediapipe anthropic opencv-python-headless matplotlib numpy

## 2. API key (Colab secrets or env)

Recommended: **Colab** → **Secrets** → add `ANTHROPIC_API_KEY`. Or set the variable in the next cell.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass

assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY (Colab secret or os.environ)"

## 3. Provide video path

- **Upload:** use Colab file upload widget or mount Drive and set `VIDEO_PATH`.
- **Sample clip:** any MP4 with a visible face (phone selfie video works for a smoke test).

In [ ]:
from google.colab import files

VIDEO_PATH = "sample.mp4"  # change after upload

uploaded = files.upload()  # optional: run once, then set VIDEO_PATH to uploaded name
if uploaded:
    VIDEO_PATH = list(uploaded.keys())[0]
    print("Using", VIDEO_PATH)

## 4. Landmark metrics (aligned with browser DMS)

Same 6-point EAR/MAR indices and geometric head pose as `static/js/vision.js`.

In [ ]:
import math
from dataclasses import dataclass
from typing import List, Tuple

import cv2
import mediapipe as mp
import numpy as np

LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]
MOUTH_INNER = [78, 81, 13, 311, 308, 402]


def dist2(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])


def aspect_ratio_6(lm, idxs, w, h):
    pts = [(lm[i].x * w, lm[i].y * h) for i in idxs]
    p1, p2, p3, p4, p5, p6 = pts
    h0 = dist2(p1, p4)
    if h0 < 1e-6:
        return 0.0
    return (dist2(p2, p6) + dist2(p3, p5)) / (2 * h0)


def head_pose_geometric(lm, w, h):
    le = (lm[33].x * w, lm[33].y * h)
    re = (lm[263].x * w, lm[263].y * h)
    nose = (lm[1].x * w, lm[1].y * h)
    chin = (lm[152].x * w, lm[152].y * h)
    mid = ((le[0] + re[0]) / 2, (le[1] + re[1]) / 2)
    roll = math.degrees(math.atan2(re[1] - le[1], re[0] - le[0]))
    face_w = max(dist2(le, re), 1e-6)
    yaw = -math.degrees(math.atan2(nose[0] - mid[0], face_w * 0.35))
    pitch = -math.degrees(
        math.atan2(nose[1] - mid[1], abs(chin[1] - nose[1]) + 1e-6)
    )
    return {"yaw": yaw, "pitch": pitch, "roll": roll}


@dataclass
class FrameMetrics:
    frame: int
    t_sec: float
    ear_left: float
    ear_right: float
    mar: float
    yaw: float
    pitch: float
    roll: float


def process_video(path: str, stride: int = 2) -> Tuple[List[FrameMetrics], float]:
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    mesh = mp.solutions.face_mesh.FaceMesh(
        static_image_mode=False,
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    out: List[FrameMetrics] = []
    frame_i = 0
    while True:
        ok, bgr = cap.read()
        if not ok:
            break
        if frame_i % stride != 0:
            frame_i += 1
            continue
        h, w = bgr.shape[:2]
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        res = mesh.process(rgb)
        t_sec = frame_i / fps
        if res.multi_face_landmarks:
            lm = res.multi_face_landmarks[0].landmark
            el = aspect_ratio_6(lm, LEFT_EYE, w, h)
            er = aspect_ratio_6(lm, RIGHT_EYE, w, h)
            mar = aspect_ratio_6(lm, MOUTH_INNER, w, h)
            hp = head_pose_geometric(lm, w, h)
            out.append(
                FrameMetrics(
                    frame_i,
                    t_sec,
                    el,
                    er,
                    mar,
                    hp["yaw"],
                    hp["pitch"],
                    hp["roll"],
                )
            )
        frame_i += 1
    cap.release()
    mesh.close()
    return out, fps

In [ ]:
frames, FPS = process_video(VIDEO_PATH, stride=2)
print(f"Extracted {len(frames)} face frames (stride=2), video FPS≈{FPS:.1f}")

## 5. Signal summary + Claude (debounced, mirrors production intent)

Rolling window of the last **K** frames → text summary → Claude JSON `{severity, alert_text, reasoning}`. Debounce: only call when summary string changes or **30s** video time elapsed since last call.

In [ ]:
import hashlib
import json
import time

from anthropic import Anthropic

SYSTEM = (
    "You are a driver safety co-pilot. Given physiological signals, assess drowsiness "
    "severity (none/mild/moderate/severe) and produce a short alert. "
    "Respond ONLY as JSON: {severity, alert_text, reasoning}"
)

MAR_YAWN = 0.42
EAR_LOW = 0.19
EAR_DROWSY = 0.21
WINDOW = 20  # frames in rolling summary


def count_yawns(mar_vals):
    ev = in_spike = 0
    for v in mar_vals:
        if v >= MAR_YAWN:
            if not in_spike:
                ev += 1
                in_spike = True
        elif v < MAR_YAWN * 0.75:
            in_spike = False
    return min(ev, 5)


def build_summary(chunk: List[FrameMetrics]) -> str:
    ears = [(f.ear_left + f.ear_right) / 2 for f in chunk]
    n = len(ears)
    ear_avg = sum(ears) / n
    below = sum(1 for e in ears if e < EAR_DROWSY)
    mar_vals = [f.mar for f in chunk]
    mar_avg = sum(mar_vals) / n
    mar_max = max(mar_vals)
    pitches = [f.pitch for f in chunk]
    nod = min(pitches) < -12 or (max(pitches) - min(pitches)) > 18
    parts = [
        f"EAR avg {ear_avg:.2f} over {n} frames",
        f"MAR avg {mar_avg:.2f}, peak {mar_max:.2f}",
    ]
    y = count_yawns(mar_vals)
    if y:
        parts.append(f"{y} yawn-like mouth events")
    if nod:
        parts.append("head nodding / pitch swing")
    if ear_avg < EAR_LOW:
        parts.append("low EAR")
    if below / n >= 0.5:
        parts.append(f"EAR<{EAR_DROWSY} in {below/n:.0%} of frames")
    parts.append("highway road, 60 mph, 30 min session (simulated context)")
    return ", ".join(parts)


def parse_json(text: str):
    t = text.strip()
    if t.startswith("```"):
        t = t.split("\n", 1)[-1].rsplit("```", 1)[0].strip()
    try:
        return json.loads(t)
    except json.JSONDecodeError:
        return None


def claude_alert(client: Anthropic, summary: str):
    msg = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=256,
        system=SYSTEM,
        messages=[{"role": "user", "content": f"Signal summary:\n{summary}\n\nJSON only."}],
    )
    text = "".join(b.text for b in msg.content if hasattr(b, "text"))
    data = parse_json(text) or {}
    sev = str(data.get("severity", "none")).lower()
    if sev not in ("none", "mild", "moderate", "severe"):
        sev = "none"
    return sev, str(data.get("alert_text", "")), data.get("reasoning")


def run_claude_engine(frames: List[FrameMetrics]):
    client = Anthropic()
    events = []
    last_hash = ""
    last_t = -1e9
    debounce_sec = 30.0
    for i in range(WINDOW, len(frames)):
        chunk = frames[i - WINDOW : i]
        summary = build_summary(chunk)
        key = hashlib.sha256(summary.encode()).hexdigest()[:16]
        t_video = chunk[-1].t_sec
        if key == last_hash and (t_video - last_t) < debounce_sec:
            continue
        last_hash = key
        last_t = t_video
        sev, alert, reason = claude_alert(client, summary)
        if sev == "none":
            continue
        f = chunk[-1]
        ear = (f.ear_left + f.ear_right) / 2
        events.append(
            {
                "frame": f.frame,
                "timestamp": f.t_sec,
                "ear": ear,
                "mar": f.mar,
                "head_pose": {"yaw": f.yaw, "pitch": f.pitch, "roll": f.roll},
                "severity": sev,
                "alert_text": alert,
                "reasoning": reason,
                "engine": "claude",
            }
        )
    return events

## 6. Fixed-threshold baseline (resume comparison)

Fire an alert when **mean EAR** over a 1-second sliding window is **< 0.21** (window length = `int(fps)` frames at stride 1 equivalent; here we scale by stride).

In [ ]:
def run_fixed_baseline(frames: List[FrameMetrics], fps: float, stride: int = 2):
    win = max(3, int(fps / stride))  # ~1 second of processed frames
    events = []
    for i in range(win, len(frames)):
        chunk = frames[i - win : i]
        ears = [(f.ear_left + f.ear_right) / 2 for f in chunk]
        if sum(ears) / len(ears) < 0.21:
            f = chunk[-1]
            events.append(
                {
                    "frame": f.frame,
                    "timestamp": f.t_sec,
                    "ear": ears[-1],
                    "mar": f.mar,
                    "head_pose": {"yaw": f.yaw, "pitch": f.pitch, "roll": f.roll},
                    "severity": "baseline",
                    "alert_text": "Fixed threshold: sustained low EAR",
                    "engine": "fixed_ear_021",
                }
            )
    return events

In [ ]:
claude_events = run_claude_engine(frames)
baseline_events = run_fixed_baseline(frames, FPS, stride=2)

n_b = len(baseline_events)
n_c = len(claude_events)
reduction_pct = (1 - n_c / n_b) * 100 if n_b else 0.0

print("Baseline (fixed EAR) alert rows:", n_b)
print("Claude engine alert rows:", n_c)
print(f"Alert reduction vs baseline: {reduction_pct:.1f}% (placeholder for resume — depends on clip)")

from collections import Counter

sev_c = Counter(e["severity"] for e in claude_events)
print("Claude severity distribution:", dict(sev_c))

## 7. Merge log + summary stats

In [ ]:
all_log = sorted(claude_events + baseline_events, key=lambda x: (x["timestamp"], x["engine"]))

summary = {
    "video": VIDEO_PATH,
    "face_frames": len(frames),
    "baseline_alert_count": n_b,
    "claude_alert_count": n_c,
    "pct_fewer_alerts_than_baseline": round(reduction_pct, 2),
    "claude_severity_counts": dict(sev_c),
}

with open("benchmark_log.json", "w") as f:
    json.dump({"summary": summary, "events": all_log}, f, indent=2)
print("Wrote benchmark_log.json")

## 8. Plots: EAR / MAR with alert markers

In [ ]:
import matplotlib.pyplot as plt

ts = [f.t_sec for f in frames]
ear = [(f.ear_left + f.ear_right) / 2 for f in frames]
mar = [f.mar for f in frames]

fig, ax = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
ax[0].plot(ts, ear, color="#38bdf8", lw=1)
ax[0].axhline(0.21, color="orange", ls="--", lw=1, label="EAR 0.21")
ax[0].set_ylabel("EAR")
ax[0].legend(loc="upper right")
for e in baseline_events[:: max(1, len(baseline_events) // 200)]:
    ax[0].axvline(e["timestamp"], color="red", alpha=0.08)

ax[1].plot(ts, mar, color="#a78bfa", lw=1)
ax[1].set_ylabel("MAR")
ax[1].set_xlabel("time (s)")
for e in claude_events:
    ax[1].axvline(e["timestamp"], color="lime", alpha=0.25, lw=1.2)

plt.suptitle("Blue=EAR, purple=MAR; faint red=baseline hits, green=Claude alerts")
plt.tight_layout()
plt.savefig("benchmark_plots.png", dpi=150)
plt.show()

## 9. Resume wording (fill in numbers from `summary`)

- "Built a real-time driver monitoring web app (MediaPipe + FastAPI + WebSockets) with **context-aware** Claude alerts."
- "Benchmarked on **[dataset / N clips]**: contextual engine produced **__%** fewer alerts than a fixed EAR<0.21s baseline while preserving safety-relevant events (manual review / labels: **TBD**)."

**Note:** "False alert" rate needs ground-truth drowsiness labels. NTHU-DDD / UTA-RLDD provide research labels in various formats — align this notebook’s `baseline_events` / `claude_events` timestamps with those labels for a rigorous false-positive rate.